# 17 Compute stock accounting features

目标：计算 30 个会计、投资、盈利、价值、杠杆与 AT 特征，包括严格事件筛选的 D2P。

- 输入：15 号 Compustat/CCM、14 号月度股票、Distribution
- 正式输出：stock_accounting_features_raw.parquet、D2P 事件审计
- 主键：permno+month
- 上游依赖：14、15
- smoke 模式仅使用真实本地数据子样本，并写入 `data/audit/smoke/`；不冒充正式结果。

In [ ]:
from __future__ import annotations
import json, os, sys, time
from pathlib import Path

ROOT = Path.cwd().resolve()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from src.config_utils import load_project_config
from src.path_utils import find_project_root
from src.runtime_paths import (
    clean_output_path,
    execution_context,
    install_clean_read_guard,
)

ROOT = find_project_root(ROOT)
EXECUTION_CONTEXT = execution_context()
install_clean_read_guard(ROOT)
CONFIG = load_project_config(ROOT)
PARAMS = CONFIG["parameters"]
PATHS = CONFIG["resolved_paths"]
RUN_MODE = os.getenv("KANIEL_RUN_MODE", "smoke").strip().lower()
SMOKE = RUN_MODE != "full"
if RUN_MODE == "full":
    FULL_RUN_TMP = (
        clean_output_path(ROOT, "data/interim/_full_run_tmp")
        if EXECUTION_CONTEXT == "clean_rebuild_compute"
        else PATHS["full_run_tmp"]
    )
    PATHS["full_run_tmp"] = FULL_RUN_TMP
    FULL_RUN_TMP.mkdir(parents=True, exist_ok=True)
    os.environ["TMP"] = str(FULL_RUN_TMP)
    os.environ["TEMP"] = str(FULL_RUN_TMP)
    os.environ["TMPDIR"] = str(FULL_RUN_TMP)
    os.environ["POLARS_TEMP_DIR"] = str(FULL_RUN_TMP)
STARTED_AT = time.time()
print({
    "project_root": str(ROOT),
    "run_mode": RUN_MODE,
    "execution_context": EXECUTION_CONTEXT,
})


In [ ]:
import numpy as np
import pandas as pd
from src.atomic_write import write_csv_atomic, write_parquet_atomic
from src.runtime_paths import output_path, upstream_path

comp=pd.read_parquet(upstream_path(ROOT,RUN_MODE,"15_prepare_compustat_and_ccm","data/processed/stock/compustat_ccm_mapped.parquet"))
stock=pd.read_parquet(upstream_path(ROOT,RUN_MODE,"14_prepare_crsp_stock_monthly_daily","data/processed/stock/stock_monthly_clean.parquet"))
comp["gvkey"]=comp["gvkey"].astype(str).str.replace(r"\.0$","",regex=True).str.zfill(6)
for c in ["datadate","linkdt","linkenddt"]: 
    if c in comp: comp[c]=pd.to_datetime(comp[c])
stock["month"]=pd.to_datetime(stock["month"])
comp=comp.sort_values(["gvkey","datadate"])
lagcols=["at","ppegt","invt","csho","ajex","BE","ceq","che","ivao","dlc","dltt","mib","pstk","act","lct","txp"]
for c in lagcols: comp[c+"_lag"]=comp.groupby("gvkey")[c].shift(1)
def div(a,b,positive=False):
    x=a/b; return x.where(b.gt(0) if positive else b.ne(0))
comp["noa_dollar"]=comp["at"]-comp["che"]-comp["ivao"]-(comp["at"]-comp["dlc"]-comp["dltt"]-comp["mib"]-comp["pstk"]-comp["ceq"])
comp["noa_dollar_lag"]=comp.groupby("gvkey")["noa_dollar"].shift(1)
me_dec=stock[stock["month"].dt.month==12][["permno","month","market_equity_thousand"]].copy()
me_dec["year"]=me_dec["month"].dt.year; me_dec["me_dec_million"]=me_dec["market_equity_thousand"]/1000.0
me_jun=stock[stock["month"].dt.month==6][["permno","month","market_equity_thousand"]].copy()
me_jun["formation_year"]=me_jun["month"].dt.year
me_jun["me_jun_million"]=me_jun["market_equity_thousand"]/1000.0

# Notebook 15 already applies the locked CCM selection at gvkey+datadate.
mapped=comp.copy()
mapped["formation_year"]=mapped["datadate"].dt.year+1
mapped["prior_year"]=mapped["formation_year"]-1
mapped=mapped.merge(me_dec[["permno","year","me_dec_million"]],left_on=["permno","prior_year"],right_on=["permno","year"],how="left")
mapped=mapped.merge(me_jun[["permno","formation_year","me_jun_million"]],on=["permno","formation_year"],how="left")
mapped["Investment"]=div(mapped["at"]-mapped["at_lag"],mapped["at_lag"],True)
mapped["NOA"]=div(mapped["noa_dollar"],mapped["at_lag"],True)
mapped["DPI2A"]=div((mapped["ppegt"]-mapped["ppegt_lag"])+(mapped["invt"]-mapped["invt_lag"]),mapped["at_lag"],True)
shares=mapped["csho"]*mapped["ajex"]; shares_lag=mapped["csho_lag"]*mapped["ajex_lag"]
mapped["NI"]=np.log(shares.where(shares>0))-np.log(shares_lag.where(shares_lag>0))
mapped["PROF"]=div(mapped["gp"],mapped["BE"],True)
mapped["ATO"]=div(mapped["sale"],mapped["noa_dollar_lag"])
mapped["CTO"]=div(mapped["sale"],mapped["at_lag"],True)
mapped["FC2Y"]=div(mapped["xsga"]+mapped["xrd"]+mapped["xad"],mapped["sale"],True)
mapped["OP"]=div(mapped["revt"]-mapped["cogs"]-mapped["xint"]-mapped["xsga"],mapped["BE"],True)
mapped["PM"]=div(mapped["oiadp"],mapped["sale"])
mapped["RNA"]=div(mapped["oiadp"],mapped["noa_dollar_lag"])
mapped["ROA"]=div(mapped["ib"],mapped["at_lag"],True); mapped["ROE"]=div(mapped["ib"],mapped["BE_lag"],True)
mapped["SGA2S"]=div(mapped["xsga"],mapped["sale"]); mapped["D2A"]=div(mapped["dp"],mapped["at"],True)
owc_now=mapped["act"]-mapped["che"]-mapped["lct"]-mapped["dlc"]-mapped["txp"]
owc_prev=mapped.groupby("gvkey")[["act","che","lct","dlc","txp"]].shift(1)
owc_prev=owc_prev["act"]-owc_prev["che"]-owc_prev["lct"]-owc_prev["dlc"]-owc_prev["txp"]
mapped["AC"]=div(div(owc_now, mapped["csho"]*mapped["ajex"])-div(owc_prev,mapped["csho_lag"]*mapped["ajex_lag"]),div(mapped["BE"],mapped["csho"]*mapped["ajex"]))
mapped["OA"]=div((owc_now-owc_prev)-mapped["dp"],mapped["at_lag"],True)
mapped["OL"]=div(mapped["cogs"]+mapped["xsga"],mapped["at"],True); mapped["PCM"]=div(mapped["sale"]-mapped["cogs"],mapped["sale"])
mapped["A2ME"]=div(mapped["at"],mapped["me_dec_million"],True); mapped["BEME"]=div(mapped["BE"],mapped["me_dec_million"],True)
mapped["C"]=div(mapped["che"],mapped["at"],True); mapped["CF"]=div(mapped["ni"]+mapped["dp"]-mapped["wcapch"]-mapped["capx"],mapped["BE"],True)
mapped["CF2P"]=div(mapped["ib"]+mapped["dp"]+mapped["txdb"],mapped["me_dec_million"],True)
mapped["E2P"]=div(mapped["ib"],mapped["me_dec_million"],True); mapped["Q"]=div(mapped["at"]+mapped["me_dec_million"]-mapped["ceq"]-mapped["txdb"],mapped["at"],True)
mapped["S2P"]=div(mapped["sale"],mapped["me_dec_million"],True)
mapped["Lev"]=div(mapped["dltt"]+mapped["dlc"],mapped["dltt"]+mapped["dlc"]+mapped["seq"]); mapped["AT"]=mapped["at"]

dist=pd.read_csv(PATHS["distribution"],nrows=None if not SMOKE else 300_000,low_memory=False)
dist=dist.rename(columns={"PERMNO":"permno","DisExDt":"disexdt","DisSeqNbr":"disseqnbr","DisType":"distype","DisPaymentType":"dispaymenttype","DisDetailType":"disdetailtype","DisDivAmt":"disdivamt","DisOrdinaryFlg":"disordinaryflg"})
dist["disexdt"]=pd.to_datetime(dist["disexdt"],errors="coerce"); dist["disdivamt"]=pd.to_numeric(dist["disdivamt"],errors="coerce")
dist=dist.drop_duplicates(["permno","disexdt","disseqnbr"])
valid=dist["distype"].isin(["CD","SD"])&dist["dispaymenttype"].eq("USD")&dist["disdetailtype"].isin(["CDIV","SDIV"])
excluded_codes=["ROC","SDROC","CDM","CDPSR"]
valid&=~dist["disdetailtype"].isin(excluded_codes)&dist["disdivamt"].gt(0)&np.isfinite(dist["disdivamt"])
d2audit=dist.loc[~valid].assign(reason_code="not_locked_cash_dividend")
cash=dist.loc[valid].copy(); cash["formation_year"]=np.where(cash["disexdt"].dt.month>=7,cash["disexdt"].dt.year+1,cash["disexdt"].dt.year)
cash=cash.groupby(["permno","formation_year"],as_index=False)["disdivamt"].sum().rename(columns={"disdivamt":"cash_dividend"})
mapped=mapped.merge(cash,on=["permno","formation_year"],how="left")
mapped["D2P"]=div(mapped["cash_dividend"],mapped["me_jun_million"],True)

feature_names=['Investment', 'NOA', 'DPI2A', 'NI', 'PROF', 'ATO', 'CTO', 'FC2Y', 'OP', 'PM', 'RNA', 'ROA', 'ROE', 'SGA2S', 'D2A', 'AC', 'OA', 'OL', 'PCM', 'A2ME', 'BEME', 'C', 'CF', 'CF2P', 'D2P', 'E2P', 'Q', 'S2P', 'Lev', 'AT']
mapped=mapped.loc[mapped["permno"].notna()&mapped["formation_year"].notna()].copy()
base_features=mapped[["permno","formation_year"]+feature_names].reset_index(drop=True)
repeat_index=np.repeat(np.arange(len(base_features)),12)
month_offset=np.tile(np.arange(12),len(base_features))
features=base_features.iloc[repeat_index].reset_index(drop=True)
features["month"]=(
    pd.to_datetime(features["formation_year"].astype(int).astype(str)+"-07-01")
    + pd.to_timedelta(0,unit="D")
)
features["month"]=features["month"].dt.to_period("M")+month_offset
features["month"]=features["month"].dt.to_timestamp()
features=features[["permno","month"]+feature_names].sort_values(["permno","month"]).drop_duplicates(["permno","month"],keep="last")
assert list(features.columns)==["permno","month"]+feature_names
out=output_path(ROOT,RUN_MODE,"17_compute_stock_accounting_features","data/processed/stock/stock_accounting_features_raw.parquet")
aud=output_path(ROOT,RUN_MODE,"17_compute_stock_accounting_features","data/audit/17_d2p_event_exclusions.csv")
aud_missing=output_path(ROOT,RUN_MODE,"17_compute_stock_accounting_features","data/audit/17_accounting_feature_missing_reasons.parquet")
missing_summary=pd.DataFrame([{
    "feature":c,"n_rows":len(features),"n_valid":int(features[c].notna().sum()),
    "n_missing":int(features[c].isna().sum()),
    "missing_rate":float(features[c].isna().mean()),
    "missing_reason":"formula_component_or_denominator_unavailable",
} for c in feature_names])
write_parquet_atomic(features,out); write_csv_atomic(d2audit,aud); write_parquet_atomic(missing_summary,aud_missing)
INPUTS_USED=["compustat_ccm_mapped.parquet","stock_monthly_clean.parquet",PATHS["distribution"].name]
OUTPUTS_WRITTEN=[str(p.relative_to(ROOT)) for p in [out,aud,aud_missing]]; INPUT_COUNTS={"compustat":len(comp),"distribution":len(dist)}; OUTPUT_COUNTS={"features":len(features)}
PRIMARY_KEY=["permno","month"]; DUPLICATE_KEY_COUNT=int(features.duplicated(PRIMARY_KEY).sum()); EXCLUDED_ROW_COUNT=len(d2audit); EXCEPTION_FILE=str(aud.relative_to(ROOT))
STAGE_STATUS="passed"; STAGE_NOTES=["FC2Y does not fill XRD or XAD with zero.","D2P uses CD/CDIV and SD/SDIV, USD payment, excludes non-cash/ROC categories, and uses formation-year June market equity.","Accounting statements are expanded from formation-year July through the following June without future information."]


In [ ]:
from src.audit_utils import write_run_summary
summary_path = write_run_summary(
    ROOT, "17_compute_stock_accounting_features", stage="17", started_at=STARTED_AT,
    input_files=INPUTS_USED, output_files=OUTPUTS_WRITTEN,
    input_row_counts=INPUT_COUNTS, output_row_counts=OUTPUT_COUNTS,
    primary_key=PRIMARY_KEY, duplicate_key_count=DUPLICATE_KEY_COUNT,
    excluded_row_count=EXCLUDED_ROW_COUNT, exception_file=EXCEPTION_FILE,
    status=STAGE_STATUS, notes=STAGE_NOTES,
)
print(summary_path)
